# Credit Card Fraud Detection — Step 9: Focal Loss & Hard Example Mining from Scratch
### Milestone 9 of the 30-Day Capstone Upgrade
**Course:** Python for Data Science (PDS)
**Focus:** Mathematical Derivation of Focal Loss, Analytical Gradients for Imbalanced Learning, and Hard-Example Mining in Pure NumPy.
**Constraint:** Strictly zero `sklearn` or external ML packages — 100% pure NumPy, Pandas, and Matplotlib.

---

## 1. Theoretical Foundations & Mathematical Formulations

### 1.1 The Gradient Swamping Pathology in Extreme Imbalance
In credit card fraud detection, genuine transactions account for $>99.8\%$ of all records. Under standard Binary Cross-Entropy (BCE):
$$\mathcal{L}_{\text{BCE}}(p_t) = -\ln(p_t)$$
where $p_t = \hat{y}$ if $y=1$, and $p_t = 1 - \hat{y}$ if $y=0$.

Even when an easy genuine transaction is correctly classified with high confidence ($p_t = 0.99$), it still produces a non-zero loss of $-\ln(0.99) \approx 0.01$. When summed over **284,000+ genuine records**, these trivial errors aggregate to over **$2,800$ units of cumulative loss**, completely swamping the sparse gradient signals generated by the 492 fraud instances.

### 1.2 Mathematical Formulation of Focal Loss
Focal Loss (Lin et al., ICCV 2017) introduces a modulating factor $(1 - p_t)^\gamma$ with focusing parameter $\gamma \ge 0$ and class-weighting factor $\alpha_t \in (0, 1)$:
$$\mathcal{L}_{\text{FL}}(p_t) = -\alpha_t (1 - p_t)^\gamma \ln(p_t)$$
where:
$$p_t = \begin{cases} \hat{y}, & \text{if } y = 1 \\ 1 - \hat{y}, & \text{if } y = 0 \end{cases}, \quad \alpha_t = \begin{cases} \alpha, & \text{if } y = 1 \\ 1 - \alpha, & \text{if } y = 0 \end{cases}$$

#### Modulating Properties:
- When $\gamma = 0$, Focal Loss is identical to standard Binary Cross-Entropy: $(1 - p_t)^0 = 1$.
- When $\gamma = 2$ and an easy example has $p_t = 0.99$, the modulating factor is $(1 - 0.99)^2 = \mathbf{0.0001}$, **reducing its loss and gradient by a factor of 10,000**.
- For a hard, ambiguous fraud instance with $p_t = 0.50$, the modulating factor is $(1 - 0.50)^2 = \mathbf{0.25}$ — receiving **2,500\times more optimization focus** than the easy sample.

### 1.3 Analytical Gradient Derivation via Multivariate Chain Rule
Let $z = \mathbf{w}^T \mathbf{x} + b$ and $\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$. Note the derivative identity $\frac{\partial \hat{y}}{\partial z} = \hat{y}(1 - \hat{y})$.

#### Case 1: Positive Sample ($y = 1, p_t = \hat{y}$)
$$\mathcal{L}_1 = -\alpha (1 - \hat{y})^\gamma \ln(\hat{y})$$
$$\frac{\partial \mathcal{L}_1}{\partial \hat{y}} = \alpha \left[ \gamma (1 - \hat{y})^{\gamma - 1} \ln(\hat{y}) - \frac{(1 - \hat{y})^\gamma}{\hat{y}} \right]$$
$$\frac{\partial \mathcal{L}_1}{\partial z} = \frac{\partial \mathcal{L}_1}{\partial \hat{y}} \cdot \hat{y}(1 - \hat{y}) = \alpha (1 - \hat{y})^\gamma \left[ \gamma \hat{y} \ln(\hat{y}) + \hat{y} - 1 \right]$$

#### Case 2: Negative Sample ($y = 0, p_t = 1 - \hat{y}$)
$$\mathcal{L}_0 = -(1 - \alpha) \hat{y}^\gamma \ln(1 - \hat{y})$$
$$\frac{\partial \mathcal{L}_0}{\partial \hat{y}} = -(1 - \alpha) \left[ \gamma \hat{y}^{\gamma - 1} \ln(1 - \hat{y}) - \frac{\hat{y}^\gamma}{1 - \hat{y}} \right]$$
$$\frac{\partial \mathcal{L}_0}{\partial z} = \frac{\partial \mathcal{L}_0}{\partial \hat{y}} \cdot \hat{y}(1 - \hat{y}) = (1 - \alpha) \hat{y}^\gamma \left[ -\gamma (1 - \hat{y}) \ln(1 - \hat{y}) + \hat{y} \right]$$

Defining $g_i = \frac{\partial \mathcal{L}_{\text{FL}}^{(i)}}{\partial z^{(i)}}$, the full gradient vectors with $L_2$ regularization are:
$$\frac{\partial \mathcal{L}_{\text{FL}}}{\partial \mathbf{w}} = \frac{1}{m} \mathbf{X}^T \mathbf{g} + \frac{\lambda}{m} \mathbf{w}, \quad \frac{\partial \mathcal{L}_{\text{FL}}}{\partial b} = \frac{1}{m} \sum_{i=1}^m g_i$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Seed & Configuration
SEED = 42
np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Pure NumPy environment initialized for Step 9 Focal Loss.')

## 2. Visualizing Theoretical Focal Loss Modulation vs. Standard Cross-Entropy
Demonstrating how varying $\gamma$ silences well-classified examples ($p_t > 0.5$).

In [ ]:
pt = np.linspace(0.001, 1.0, 500)
gammas = [0.0, 0.5, 1.0, 2.0, 5.0]
colors = ['#1f77b4', '#aec7e8', '#ff7f0e', '#d62728', '#9467bd']

plt.figure(figsize=(10, 5))
for g, col in zip(gammas, colors):
    fl = -(1.0 - pt)**g * np.log(pt)
    label = f'gamma = {g} (BCE)' if g == 0.0 else f'gamma = {g}'
    plt.plot(pt, fl, label=label, color=col, lw=2.2)

plt.xlabel('Probability of Ground Truth Class ($p_t$)')
plt.ylabel('Focal Loss Value')
plt.title('Focal Loss Curves: Exponential Down-Weighting of Easy Examples ($p_t \to 1$)')
plt.ylim(0, 5)
plt.xlim(0, 1)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## 3. Dataset Ingestion, Cyclic Time Encoding & Stratified Split

In [ ]:
df = pd.read_csv('../data/creditcard.csv')

# Feature Engineering: Cyclic 24h Diurnal Encoding + Log-Amount
seconds_in_day = 86400.0
time_sin = np.sin(2 * np.pi * df['Time'].values / seconds_in_day)
time_cos = np.cos(2 * np.pi * df['Time'].values / seconds_in_day)
log_amount = np.log1p(df['Amount'].values)
v_cols = [f'V{i}' for i in range(1, 29)]
X_all = np.column_stack([time_sin, time_cos, log_amount, df[v_cols].values])
y_all = df['Class'].values.astype(np.int64)
amounts_all = df['Amount'].values

# Stratified 80/20 train/test split from scratch
idx_0 = np.where(y_all == 0)[0]
idx_1 = np.where(y_all == 1)[0]
np.random.shuffle(idx_0)
np.random.shuffle(idx_1)
n_te_0 = int(len(idx_0) * 0.2)
n_te_1 = int(len(idx_1) * 0.2)
test_idx = np.concatenate([idx_0[:n_te_0], idx_1[:n_te_1]])
train_idx = np.concatenate([idx_0[n_te_0:], idx_1[n_te_1:]])
np.random.shuffle(test_idx)
np.random.shuffle(train_idx)

X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]
amounts_train, amounts_test = amounts_all[train_idx], amounts_all[test_idx]

# Note: Unlike standard BCE which requires 50/50 undersampling,
# Focal Loss can handle much higher class imbalance (e.g. 10:1 or 20:1)
# Here we test on balanced data to isolate the gradient effect of gamma
idx_0_tr = np.where(y_train == 0)[0]
idx_1_tr = np.where(y_train == 1)[0]
sampled_0 = np.random.choice(idx_0_tr, size=len(idx_1_tr) * 4, replace=False)

X_tr_sub = np.vstack([X_train[idx_1_tr], X_train[sampled_0]])
y_tr_sub = np.concatenate([np.ones(len(idx_1_tr), dtype=np.int64), np.zeros(len(sampled_0), dtype=np.int64)])
perm = np.random.permutation(len(y_tr_sub))
X_tr_sub, y_tr_sub = X_tr_sub[perm], y_tr_sub[perm]

mean = np.mean(X_tr_sub, axis=0)
std = np.std(X_tr_sub, axis=0)
std[std == 0.0] = 1.0
X_tr_norm = (X_tr_sub - mean) / std
X_te_norm = (X_test - mean) / std

print(f'Training Samples: {len(X_tr_norm):,} (80% genuine, 20% fraud)')
print(f'Held-Out Test   : {len(X_te_norm):,} samples (with ${amounts_test[y_test==1].sum():,.2f} fraud)')

## 4. Pure NumPy Implementation of Focal Loss Classifier from Scratch
Implements the exact derived analytical gradient expressions with numerical stabilization.

In [ ]:
class FocalLossLogisticRegressionScratch:
    """
    First-Principles Binary Classifier trained via Focal Loss:
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * ln(p_t)
    Uses analytical gradient vectors derived via chain rule.
    """
    def __init__(self, gamma=2.0, alpha=0.75, lr=0.1, n_iter=800, l2_reg=0.01):
        self.gamma = gamma
        self.alpha = alpha
        self.lr = lr
        self.n_iter = n_iter
        self.l2_reg = l2_reg
        self.w = None
        self.b = 0.0
        self.loss_history = []
        
    def _compute_loss_and_gradients(self, X, y, w, b):
        m = len(y)
        z = np.clip(np.dot(X, w) + b, -500.0, 500.0)
        y_hat = 1.0 / (1.0 + np.exp(-z))
        eps = 1e-15
        y_hat = np.clip(y_hat, eps, 1.0 - eps)
        
        # Vectorized Gradient Computation based on derived chain-rule formulas
        # For y = 1: g_1 = alpha * (1 - y_hat)^gamma * [gamma * y_hat * ln(y_hat) + y_hat - 1]
        # For y = 0: g_0 = (1 - alpha) * y_hat^gamma * [-gamma * (1 - y_hat) * ln(1 - y_hat) + y_hat]
        
        g = np.zeros(m, dtype=np.float64)
        mask_1 = (y == 1)
        mask_0 = ~mask_1
        
        if self.gamma == 0.0:
            # Standard weighted BCE gradient: alpha_t * (y_hat - y)
            g[mask_1] = self.alpha * (y_hat[mask_1] - 1.0)
            g[mask_0] = (1.0 - self.alpha) * y_hat[mask_0]
            loss = -np.mean(self.alpha * y * np.log(y_hat) + (1.0 - self.alpha) * (1.0 - y) * np.log(1.0 - y_hat))
        else:
            # Positive samples
            yh1 = y_hat[mask_1]
            g[mask_1] = self.alpha * ((1.0 - yh1) ** self.gamma) * (self.gamma * yh1 * np.log(yh1) + yh1 - 1.0)
            
            # Negative samples
            yh0 = y_hat[mask_0]
            g[mask_0] = (1.0 - self.alpha) * (yh0 ** self.gamma) * (-self.gamma * (1.0 - yh0) * np.log(1.0 - yh0) + yh0)
            
            # Compute Total Focal Loss
            loss_1 = -self.alpha * ((1.0 - yh1) ** self.gamma) * np.log(yh1)
            loss_0 = -(1.0 - self.alpha) * (yh0 ** self.gamma) * np.log(1.0 - yh0)
            loss = (np.sum(loss_1) + np.sum(loss_0)) / m
            
        # Add L2 penalty
        loss += (self.l2_reg / (2.0 * m)) * np.sum(w ** 2)
        dw = (1.0 / m) * np.dot(X.T, g) + (self.l2_reg / m) * w
        db = (1.0 / m) * np.sum(g)
        
        return loss, dw, db
        
    def fit(self, X, y):
        m, d = X.shape
        self.w = np.zeros(d, dtype=np.float64)
        self.b = 0.0
        self.loss_history = []
        
        for _ in range(self.n_iter):
            loss, dw, db = self._compute_loss_and_gradients(X, y, self.w, self.b)
            self.loss_history.append(loss)
            self.w -= self.lr * dw
            self.b -= self.lr * db
            
        return self
        
    def predict_proba(self, X):
        z = np.clip(np.dot(X, self.w) + self.b, -500.0, 500.0)
        return 1.0 / (1.0 + np.exp(-z))
        
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

print('FocalLossLogisticRegressionScratch class constructed successfully.')

## 5. Systematic Comparative Study: Varying $\gamma \in \{0, 1, 2, 3\}$
Evaluating how the focusing parameter shapes precision, recall, and dollar preservation on held-out test data.

In [ ]:
def trapezoid_scratch(y, x):
    return float(np.sum(0.5 * (y[:-1] + y[1:]) * np.diff(x)))

def evaluate_focal_model(y_true, amounts, prob, threshold=0.5):
    y_pred = (prob >= threshold).astype(int)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2.0 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    tot_dollars = np.sum(amounts[y_true == 1])
    caught_dollars = np.sum(amounts[(y_true == 1) & (y_pred == 1)])
    dollar_rec = caught_dollars / tot_dollars if tot_dollars > 0 else 0.0
    bank_loss = (tot_dollars - caught_dollars) + fp * 2.50
    
    # ROC-AUC via first principles
    th_s = np.linspace(0, 1, 51)
    tprs, fprs = [], []
    P, N = np.sum(y_true == 1), np.sum(y_true == 0)
    for th in th_s:
        yp = (prob >= th).astype(int)
        tprs.append(np.sum((y_true == 1) & (yp == 1)) / P)
        fprs.append(np.sum((y_true == 0) & (yp == 1)) / N)
    fpr_arr = np.array(fprs)
    tpr_arr = np.array(tprs)
    si = np.argsort(fpr_arr)
    auc = trapezoid_scratch(tpr_arr[si], fpr_arr[si])
    
    return {
        'Precision (%)': prec * 100,
        'Recall (%)': rec * 100,
        'Dollar Value Recall (%)': dollar_rec * 100,
        'F1-Score (%)': f1 * 100,
        'ROC-AUC': auc,
        'Dollars Protected ($)': caught_dollars,
        'Net Bank Loss ($)': bank_loss,
        'FP': int(fp), 'FN': int(fn)
    }

configs = [
    ('Standard BCE (gamma=0.0)', 0.0, 0.50),
    ('Mild Focal (gamma=1.0)', 1.0, 0.75),
    ('RetinaNet Focal (gamma=2.0)', 2.0, 0.75),
    ('Aggressive Focal (gamma=3.0)', 3.0, 0.85)
]

results_focal = {}
models_focal = {}

for name, g, a in configs:
    print(f'Training {name}...')
    m = FocalLossLogisticRegressionScratch(gamma=g, alpha=a, lr=0.1, n_iter=800).fit(X_tr_norm, y_tr_sub)
    p_test = m.predict_proba(X_te_norm)
    models_focal[name] = m
    results_focal[name] = evaluate_focal_model(y_test, amounts_test, p_test)

df_focal = pd.DataFrame(results_focal).T
print('\n--- FOCAL LOSS COMPARATIVE BENCHMARK ---')
df_focal[['Precision (%)', 'Recall (%)', 'Dollar Value Recall (%)', 'F1-Score (%)', 'ROC-AUC', 'Dollars Protected ($)', 'FP', 'FN']]

## 6. Visualizing Loss Convergence & Precision-Recall Dynamics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Training Loss Convergence
for name, m in models_focal.items():
    axes[0].plot(m.loss_history[:400], label=name, lw=2.0)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Focal Loss Value')
axes[0].set_title('Optimization Loss Curves across gamma Settings')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

# 2. Precision vs Dollar Value Recall
m_names = list(results_focal.keys())
precs = [results_focal[k]['Precision (%)'] for k in m_names]
rec_vals = [results_focal[k]['Dollar Value Recall (%)'] for k in m_names]
x = np.arange(len(m_names))
w = 0.35
axes[1].bar(x - w/2, precs, w, label='Precision (%)', color='#1f77b4', alpha=0.85)
axes[1].bar(x + w/2, rec_vals, w, label='Dollar Value Recall (%)', color='#2ca02c', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels([n.split(' (')[0] for n in m_names], rotation=25, ha='right')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_title('Precision vs. Capital Protection by Focal Power')
axes[1].legend()
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 7. Academic Takeaways & Methodological Insights

1. **Mathematical Solution to Gradient Domination:**
   By applying the modulating factor $(1 - p_t)^\gamma$, Focal Loss exponentially silences the millions of well-classified genuine transactions, preventing them from overwhelming the gradient trajectory.
2. **Hard-Example Automatic Mining:**
   Borderline, subtle fraud patterns ($p_t \approx 0.5$) receive substantially stronger gradient updates than clear-cut cases without needing external heuristic sampling.
3. **Zero Package Dependencies:**
   The multivariate chain-rule gradients, numerical logarithm clipping, and loss minimization were constructed 100% in pure NumPy without `scikit-learn` or `PyTorch`.